In [4]:
import torch
import sys, os

project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from peptide_pipeline.generator.vae_generator import VAEGenerator

SEQ_LENGTH = 6
INPUT_DIM = SEQ_LENGTH * 20  # 120
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")


# --- One-hot encode ---
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def encode(sequences, seq_length):
    tensor = torch.zeros(len(sequences), seq_length * 20)
    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            tensor[i, j * 20 + aa_index[aa]] = 1.0
    return tensor

In [ ]:
repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
sys.path.insert(0, repo_root)

# Import the JSONDataLoader class from the dataloader_json.py in dataloader module
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
# Example usage
json_loader = JSONDataLoader()
json_loader.load_data("database/training_data.json",columns=["sequence", "length"])

data = json_loader.get_data()

# Use the loaded data to train your model 
sequences = data["sequence"].tolist()
lengths = data["length"].tolist()
sequences.sort(key=lambda x: len(x))  # Sort sequences by length
sequences = [seq for seq in sequences if len(seq) == SEQ_LENGTH]  # Filter by desired length
print(f"Loaded {len(sequences)} sequences from JSON file.")
one_hot = encode(sequences, SEQ_LENGTH)
# --- Initialize model ---
vae = VAEGenerator(input_dim=INPUT_DIM, latent_dim=32, hidden_dim=128)
print(f"Device:     {vae.device}")
print(f"Input dim:  {vae.input_dim}")
print(f"Latent dim: {vae.latent_dim}")
print(f"Hidden dim: {vae.hidden_dim}")

# --- Train ---
one_hot = one_hot.to(vae.device)
vae.train_model(one_hot, epochs=100, batch_size=64, lr=1e-3)
print("Training complete!")

# --- Generate ---
generated = vae.generate_peptides(count=10)
amino_acid_set = set(AMINO_ACIDS)
unique_generated = set()
for i, pep in enumerate(generated):
    valid = all(c in amino_acid_set for c in pep) and len(pep) == SEQ_LENGTH
    status = "✓" if valid else "✗ INVALID"
    unique_generated.add(pep)
    print(f"Peptide {i+1:02d}: {pep}  {status}")

# --- Novelty + diversity check ---
training_set = set(sequences)
novel = [p for p in generated if p not in training_set]
print(f"\nNovel peptides  (not in training data): {len(novel)}/{len(generated)}")
print(f"Unique peptides (diversity check):      {len(unique_generated)}/{len(generated)}")

Loaded 285 sequences from JSON file.
Device:     cuda
Input dim:  120
Latent dim: 32
Hidden dim: 128
  Epoch 050/1000 | Recon: 0.2347 | KL: 1.5355 | KL weight: 0.05
  Epoch 100/1000 | Recon: 0.0578 | KL: 1.0033 | KL weight: 0.10
  Epoch 150/1000 | Recon: 0.0434 | KL: 0.7841 | KL weight: 0.15
  Epoch 200/1000 | Recon: 0.0345 | KL: 0.6995 | KL weight: 0.20
  Epoch 250/1000 | Recon: 0.0343 | KL: 0.6171 | KL weight: 0.25
  Epoch 300/1000 | Recon: 0.0520 | KL: 0.5501 | KL weight: 0.30
  Epoch 350/1000 | Recon: 0.0514 | KL: 0.5187 | KL weight: 0.35
  Epoch 400/1000 | Recon: 0.0320 | KL: 0.4973 | KL weight: 0.40
  Epoch 450/1000 | Recon: 0.0425 | KL: 0.4483 | KL weight: 0.45
  Epoch 500/1000 | Recon: 0.0478 | KL: 0.4449 | KL weight: 0.50


KeyboardInterrupt: 